In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [2]:
def send_email(recipient: str, subject: str, body: str) -> str:
    """This is the Mock function to send email."""
    return f"The email is sent to {recipient} with this subject: {subject}"

def read_email(email:str)->str:
    """this is a mock funtion to read email."""
    return f"This the summary of the email:{email}"


In [3]:
agent = create_agent(model= "groq:llama-3.1-8b-instant",
                     tools=[send_email,read_email],
                     checkpointer=InMemorySaver(),
                     middleware=[HumanInTheLoopMiddleware(
                         interrupt_on={
                             "send_email":{
                                 "allowed_decisions":["approve","edit","reject"]
                             },
                             "read_email":False
                         }
                     )],
                     
                     )

In [4]:
from langchain_core.messages import HumanMessage
config = {"configurable":{"thread_id":"test-approve"}}

result = agent.invoke(
    {"messages":[HumanMessage(content="send email to jhon@test.com with subject hello and body 'How are you?'")]},
    config = config
)

In [5]:
result

{'messages': [HumanMessage(content="send email to jhon@test.com with subject hello and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='c8e78785-4d2b-4300-acbb-c4e8400e7b91'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '04fwgmer5', 'function': {'arguments': '{"body":"How are you?","recipient":"jhon@test.com","subject":"hello"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 306, 'total_tokens': 338, 'completion_time': 0.108939541, 'completion_tokens_details': None, 'prompt_time': 0.028895012, 'prompt_tokens_details': None, 'queue_time': 0.056716278, 'total_time': 0.137834553}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e8c9b-1640-7a42-95e7-0732581ec3a6-0', tool_calls=[{'name': 'send_email', 'args': {'body': 'How are 

In [6]:
from langgraph.types import Command

In [7]:
if "__interrupt__" in result:
    print("approval pending...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type":"approve"}
                ]
            }
        ),
        config = config
    )

    print(f"Result: {result['messages'][-1].content}")

approval pending...
Result: 


In [8]:
result

{'messages': [HumanMessage(content="send email to jhon@test.com with subject hello and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='c8e78785-4d2b-4300-acbb-c4e8400e7b91'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '04fwgmer5', 'function': {'arguments': '{"body":"How are you?","recipient":"jhon@test.com","subject":"hello"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 306, 'total_tokens': 338, 'completion_time': 0.108939541, 'completion_tokens_details': None, 'prompt_time': 0.028895012, 'prompt_tokens_details': None, 'queue_time': 0.056716278, 'total_time': 0.137834553}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e8c9b-1640-7a42-95e7-0732581ec3a6-0', tool_calls=[{'name': 'send_email', 'args': {'body': 'How are 